
# TASK 1 · EDA on Retail Sales Data

**Objective:** Perform Exploratory Data Analysis (EDA) on retail sales data to uncover sales patterns, customer behaviour trends, and actionable business insights.

**Tech Stack:** Python, pandas, matplotlib, seaborn, Jupyter Notebook

### Dataset
This notebook uses the Kaggle **Retail Sales Dataset** (1,000 retail transactions) with transaction date, gender, age, age group, product category, quantity, price per unit, and total amount.

> **Important dataset limitation:** this particular dataset does **not** contain a `Product Name` column. Therefore, the notebook performs the requested product analysis at the **Product Category** level and also includes a safe, automatic `Product Name` analysis section that will run if you replace the dataset with one containing product names.


In [ ]:

# Install packages if needed:
# !pip install pandas matplotlib seaborn jupyter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Option 1: Put the downloaded CSV in the same folder as this notebook.
FILE_PATH = "Retail Sales Data Set.csv"

# Option 2: Use a Kaggle-downloaded file path instead.
# FILE_PATH = "retail_sales_dataset.csv"

df = pd.read_csv(FILE_PATH)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()


## 1. Initial Inspection
We first inspect the dataset dimensions, data types, missing values, duplicate rows, and a sample of the records.

In [ ]:

print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isnull().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())



### Observation
The dataset contains transaction-level retail records. Before analysis, we verify that dates and numerical fields are correctly typed and check for missing/duplicate records. Any missing values or duplicates should be handled before calculating business KPIs.


## 2. Data Cleaning and Feature Engineering

In [ ]:

# Clean column names
df.columns = df.columns.str.strip()

# Convert date
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Convert numerical columns
numeric_cols = ["Age", "Quantity", "Price per Unit", "Total Amount"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Create time features
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Quarter"] = df["Date"].dt.to_period("Q").astype(str)
df["Year"] = df["Date"].dt.year

print("Cleaned shape:", df.shape)
display(df[["Date", "Age", "Quantity", "Price per Unit", "Total Amount"]].describe())


### Observation
Date-based features (`Month`, `Quarter`, and `Year`) are created so that sales performance can be compared over time.

## 3. Descriptive Statistics
Mean, median, mode, and standard deviation are calculated for every numerical column.

In [ ]:

num_df = df.select_dtypes(include=np.number)

descriptive_stats = pd.DataFrame({
    "Mean": num_df.mean(),
    "Median": num_df.median(),
    "Mode": num_df.mode().iloc[0],
    "Standard Deviation": num_df.std()
})

display(descriptive_stats.round(2))



### Observation
Mean and median show the central tendency of the numeric variables, while standard deviation indicates how widely values vary. Differences between mean and median can highlight skewed transaction values or customer ages.


## 4. Monthly Sales Trend

In [ ]:

monthly_sales = (
    df.groupby("Month", as_index=True)["Total Amount"]
      .sum()
      .sort_index()
)

plt.figure(figsize=(12, 5))
monthly_sales.plot(marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

peak_month = monthly_sales.idxmax()
low_month = monthly_sales.idxmin()

print(f"Highest-sales month: {peak_month} ({monthly_sales.max():,.0f})")
print(f"Lowest-sales month: {low_month} ({monthly_sales.min():,.0f})")



### Observation
The monthly line chart shows when sales are strongest and weakest. The peak month can be used for promotional campaigns and inventory planning, while weaker months may need targeted offers or demand-generation activities.


## 5. Quarterly Sales Trend

In [ ]:

quarterly_sales = (
    df.groupby("Quarter")["Total Amount"]
      .sum()
      .sort_index()
)

plt.figure(figsize=(10, 5))
quarterly_sales.plot(marker="o")
plt.title("Quarterly Sales Trend")
plt.xlabel("Quarter")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

best_quarter = quarterly_sales.idxmax()
print(f"Highest-sales quarter: {best_quarter} ({quarterly_sales.max():,.0f})")



### Observation
Quarterly aggregation reduces daily/monthly noise and makes broader seasonal patterns easier to identify. The strongest quarter is a useful period for campaign planning, staffing, and inventory preparation.


## 6. Customer Demographics — Age Groups

In [ ]:

# Use the existing Age Group column when available; otherwise create standard groups.
if "Age Group" not in df.columns:
    bins = [17, 31, 45, 55, 65, np.inf]
    labels = ["18-30", "31-44", "45-54", "55-64", "65+"]
    df["Age Group"] = pd.cut(df["Age"], bins=bins, labels=labels)

age_group_counts = df["Age Group"].value_counts()

plt.figure(figsize=(9, 5))
age_group_counts.plot(kind="bar")
plt.title("Customer Distribution by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Largest age group:", age_group_counts.idxmax())



### Observation
The age-group chart shows which customer segment contributes the most transaction activity. This helps a retailer tailor product recommendations, advertising messages, and promotions to the largest customer segment.


## 7. Gender Breakdown

In [ ]:

gender_counts = df["Gender"].value_counts()

plt.figure(figsize=(7, 5))
gender_counts.plot(kind="bar")
plt.title("Customer Gender Breakdown")
plt.xlabel("Gender")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

gender_share = (gender_counts / gender_counts.sum() * 100).round(1)
display(gender_share.to_frame("Share (%)"))



### Observation
The gender breakdown identifies the relative contribution of each gender to transaction volume. If one group dominates, campaigns can be tailored to its preferences while maintaining offers that improve engagement among smaller segments.


## 8. Product Analysis — Top 10 Best-Selling Products

In [ ]:

# The selected Kaggle retail dataset has Product Category but no Product Name.
# This cell automatically performs product-level analysis when Product Name exists.

if "Product Name" in df.columns:
    top_products = (
        df.groupby("Product Name")["Quantity"]
          .sum()
          .sort_values(ascending=False)
          .head(10)
    )

    plt.figure(figsize=(11, 6))
    top_products.sort_values().plot(kind="barh")
    plt.title("Top 10 Best-Selling Products by Quantity")
    plt.xlabel("Units Sold")
    plt.ylabel("Product")
    plt.tight_layout()
    plt.show()

    display(top_products.to_frame("Units Sold"))
else:
    print("Product Name is not available in this dataset.")
    print("Dataset-compatible substitute: top product categories by units sold.")

    top_categories_units = (
        df.groupby("Product Category")["Quantity"]
          .sum()
          .sort_values(ascending=False)
    )

    plt.figure(figsize=(9, 5))
    top_categories_units.plot(kind="bar")
    plt.title("Best-Selling Product Categories by Units Sold")
    plt.xlabel("Product Category")
    plt.ylabel("Units Sold")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    display(top_categories_units.to_frame("Units Sold"))



### Observation
Because the chosen dataset does not contain individual product names, the fallback chart ranks product categories instead of inventing product-level results. If a product-name dataset is used, the same cell will automatically display the requested top 10 products.


## 9. Revenue by Product Category

In [ ]:

category_revenue = (
    df.groupby("Product Category")["Total Amount"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))
category_revenue.plot(kind="bar")
plt.title("Revenue by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Revenue")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(category_revenue.to_frame("Revenue").round(2))



### Observation
Revenue by category highlights the main revenue drivers. The strongest category should receive appropriate inventory priority and marketing support, while weaker categories should be reviewed for pricing, assortment, and customer-fit issues.


## 10. Correlation Heatmap

In [ ]:

corr_cols = ["Age", "Quantity", "Price per Unit", "Total Amount"]
corr = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

display(corr.round(2))



### Observation
The correlation matrix shows how numerical variables move together. A strong positive relationship between quantity and total amount is expected because transaction revenue depends directly on quantity and unit price. Correlations involving age can reveal whether spending tends to vary by customer age.


## 11. Additional Visualisation — Revenue by Age Group and Gender

In [ ]:

age_gender_revenue = (
    df.pivot_table(
        index="Age Group",
        columns="Gender",
        values="Total Amount",
        aggfunc="sum"
    )
)

plt.figure(figsize=(10, 6))
age_gender_revenue.plot(kind="bar", figsize=(10, 6))
plt.title("Revenue by Age Group and Gender")
plt.xlabel("Age Group")
plt.ylabel("Total Revenue")
plt.xticks(rotation=0)
plt.legend(title="Gender")
plt.tight_layout()
plt.show()

# Identify the highest-value age-group/gender combination
segment = (
    df.groupby(["Age Group", "Gender"])["Total Amount"]
      .sum()
      .sort_values(ascending=False)
)

print("Highest-revenue demographic segment:")
print(segment.head(1))



### Observation
This combined demographic view reveals a non-obvious insight that separate age and gender charts can hide: a particular age-group/gender combination may contribute disproportionately to revenue. This is useful for targeted campaigns and personalized offers.


## 12. Additional KPI Summary

In [ ]:

total_revenue = df["Total Amount"].sum()
total_units = df["Quantity"].sum()
avg_order_value = df["Total Amount"].mean()
avg_quantity = df["Quantity"].mean()

kpis = pd.Series({
    "Total Revenue": total_revenue,
    "Total Units Sold": total_units,
    "Average Order Value": avg_order_value,
    "Average Units per Transaction": avg_quantity,
    "Number of Transactions": len(df),
    "Unique Product Categories": df["Product Category"].nunique(),
    "Unique Customers (if available)": df["Customer ID"].nunique() if "Customer ID" in df.columns else np.nan
})

display(kpis.to_frame("Value").round(2))


## 13. Conclusion and Actionable Business Recommendations


### Key Recommendations

1. **Prioritize the highest-revenue product category.**  
   Maintain adequate inventory for the leading category and use targeted promotions to increase its contribution without creating stockouts.

2. **Target the strongest demographic segment.**  
   Use the age-group × gender analysis to personalize campaigns, recommendations, and bundled offers for the segment generating the most revenue.

3. **Plan promotions around peak sales periods.**  
   Use the monthly and quarterly trends to schedule marketing campaigns, inventory replenishment, and staffing before high-demand periods.

4. **Improve weaker customer segments.**  
   Compare low-revenue age/gender groups with the strongest segment and test targeted discounts, bundles, loyalty offers, or product recommendations.

5. **Use transaction size to design bundles.**  
   Since quantity and transaction value are directly connected, bundle complementary products or create quantity-based offers to increase average order value.



## Final Note

This notebook covers the requested EDA workflow using a Kaggle retail-sales dataset. The only checklist item that cannot be completed literally with this particular file is **Top 10 best-selling products**, because the file contains `Product Category` rather than individual `Product Name`.

For a fully literal submission, replace the input with a retail dataset containing both **customer demographics (`Age`, `Gender`) and `Product Name`**. The notebook's product-analysis cell is already designed to detect `Product Name` automatically.
